In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_train = pd.read_csv(r'F:\data science - tehran university\7 - Python\26 - 14030318\تمرین جلسه 4\Train.csv')
df_test = pd.read_csv(r'F:\data science - tehran university\7 - Python\26 - 14030318\تمرین جلسه 4\Test.csv')

In [3]:
df_train.head()

,perc_premium_paid_by_cash_credit,age_in_days,Income,Count_3.6_months_late,Count_6.12_months_late,Count_more_than_12_months_late,application_underwriting_score,no_of_premiums_paid,sourcing_channel,residence_area_type,premium,renewal
0,0.429,12058,355060,0.0,0.0,0.0,99.02,13,C,Urban,3300,1
1,0.917,17531,84140,2.0,3.0,1.0,98.69,7,C,Rural,3300,0
2,0.049,15341,250510,0.0,0.0,0.0,99.57,9,A,Urban,9600,1
3,0.052,31400,198680,0.0,0.0,0.0,99.87,12,B,Urban,9600,1
4,1.000,24829,118400,0.0,0.0,0.0,99.05,11,B,Urban,7500,1


In [4]:
df_train.dtypes

perc_premium_paid_by_cash_credit    float64
age_in_days                           int64
Income                                int64
Count_3.6_months_late               float64
Count_6.12_months_late              float64
Count_more_than_12_months_late      float64
application_underwriting_score      float64
no_of_premiums_paid                   int64
sourcing_channel                     object
residence_area_type                  object
premium                               int64
renewal                               int64
dtype: object

In [5]:
df_train['sourcing_channel'] = df_train['sourcing_channel'].astype('category')
df_train['residence_area_type'] = df_train['residence_area_type'].astype('category')
df_test['sourcing_channel'] = df_test['sourcing_channel'].astype('category')
df_test['residence_area_type'] = df_test['residence_area_type'].astype('category')

In [6]:
df_train.dtypes

perc_premium_paid_by_cash_credit     float64
age_in_days                            int64
Income                                 int64
Count_3.6_months_late                float64
Count_6.12_months_late               float64
Count_more_than_12_months_late       float64
application_underwriting_score       float64
no_of_premiums_paid                    int64
sourcing_channel                    category
residence_area_type                 category
premium                                int64
renewal                                int64
dtype: object

In [7]:
df_train.isna().sum()

perc_premium_paid_by_cash_credit       0
age_in_days                            0
Income                                 0
Count_3.6_months_late                 64
Count_6.12_months_late                64
Count_more_than_12_months_late        64
application_underwriting_score      1976
no_of_premiums_paid                    0
sourcing_channel                       0
residence_area_type                    0
premium                                0
renewal                                0
dtype: int64

In [41]:
df_train_na_filled = df_train.copy()
df_train_na_filled[['Count_3.6_months_late','Count_6.12_months_late','Count_more_than_12_months_late']] = df_train_na_filled[['Count_3.6_months_late','Count_6.12_months_late','Count_more_than_12_months_late']].fillna(-1)

In [42]:
df_test_na_filled = df_test.copy()
df_test_na_filled[['Count_3.6_months_late','Count_6.12_months_late','Count_more_than_12_months_late']] = df_test_na_filled[['Count_3.6_months_late','Count_6.12_months_late','Count_more_than_12_months_late']].fillna(-1)

In [43]:
train = df_train_na_filled[df_train_na_filled['application_underwriting_score'].isna() == False]
x_train = train.drop(['application_underwriting_score','renewal'], axis=1)
y_train = train['application_underwriting_score']

In [44]:
from sklearn.model_selection import cross_val_predict, cross_val_score, RepeatedKFold, GridSearchCV
import xgboost as xgb
model = xgb.XGBRegressor(random_state=2024, enable_categorical = True, learning_rate = 0.05, max_depth = 4, n_estimators = 300)
cv = RepeatedKFold(n_splits= 10, n_repeats= 3, random_state=2024)
scores = cross_val_score(model, x_train, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)

In [45]:
# best result
from numpy import absolute
scores = absolute(scores)
print('Mean MAE: %.3f (%.3f)' % (scores.mean(), scores.std()) )

Mean MAE: 0.397 (0.005)


In [46]:
model.fit(x_train,y_train)
prediction = df_train_na_filled[df_train_na_filled['application_underwriting_score'].isna() == True]
x_prediction = prediction.drop(['application_underwriting_score','renewal'], axis=1)
y_prediction = pd.DataFrame(model.predict(x_prediction))

In [47]:
y_prediction.index = prediction.index
y_prediction.columns=['application_underwriting_score']
df_train_completed = df_train_na_filled.copy()
df_train_completed.loc[df_train_completed['application_underwriting_score'].isna(),'application_underwriting_score'] = y_prediction
df_train_completed

,perc_premium_paid_by_cash_credit,age_in_days,Income,Count_3.6_months_late,Count_6.12_months_late,Count_more_than_12_months_late,application_underwriting_score,no_of_premiums_paid,sourcing_channel,residence_area_type,premium,renewal
0,0.429,12058,355060,0.0,0.0,0.0,99.02,13,C,Urban,3300,1
1,0.917,17531,84140,2.0,3.0,1.0,98.69,7,C,Rural,3300,0
2,0.049,15341,250510,0.0,0.0,0.0,99.57,9,A,Urban,9600,1
3,0.052,31400,198680,0.0,0.0,0.0,99.87,12,B,Urban,9600,1
4,1.000,24829,118400,0.0,0.0,0.0,99.05,11,B,Urban,7500,1
...,...,...,...,...,...,...,...,...,...,...,...,...
53231,0.994,20445,186830,0.0,0.0,0.0,99.67,5,A,Urban,18000,1
53232,0.118,22275,195070,0.0,0.0,0.0,99.25,11,A,Urban,9600,1
53233,0.033,18265,301540,0.0,0.0,0.0,99.89,4,A,Rural,13800,1
53234,0.000,23372,305020,0.0,0.0,0.0,98.89,12,A,Rural,9600,1


In [48]:
prediction2 = df_test_na_filled[df_test_na_filled['application_underwriting_score'].isna() == True]
x_prediction2 = prediction2.drop('application_underwriting_score' , axis=1)
y_prediction2 = pd.DataFrame(model.predict(x_prediction2))
y_prediction2.index = prediction2.index
y_prediction2.columns=['application_underwriting_score']
df_test_completed = df_test_na_filled.copy()
df_test_completed.loc[df_test_completed['application_underwriting_score'].isna(),'application_underwriting_score'] = y_prediction2
df_test_completed

,perc_premium_paid_by_cash_credit,age_in_days,Income,Count_3.6_months_late,Count_6.12_months_late,Count_more_than_12_months_late,application_underwriting_score,no_of_premiums_paid,sourcing_channel,residence_area_type,premium
0,0.052,14973,252030,0.0,0.0,0.0,99.38,12,D,Urban,22200
1,0.628,28117,39360,0.0,0.0,0.0,96.61,11,A,Rural,5700
2,0.458,26290,60860,0.0,0.0,0.0,99.51,6,B,Rural,1200
3,1.000,11688,66130,0.0,0.0,0.0,98.58,7,B,Urban,7500
4,0.190,25557,150140,0.0,0.0,0.0,98.48,20,B,Urban,11700
...,...,...,...,...,...,...,...,...,...,...,...
26612,0.979,14249,262590,0.0,0.0,0.0,97.24,14,A,Urban,18000
26613,0.985,12784,171700,5.0,1.0,1.0,99.49,7,E,Urban,9600
26614,0.019,21909,153790,0.0,0.0,0.0,98.74,11,A,Rural,7500
26615,0.178,13155,195150,0.0,0.0,0.0,98.90,13,D,Rural,7500


In [49]:
#df_train_completed.isna().sum()
df_test_completed.isna().sum()

perc_premium_paid_by_cash_credit    0
age_in_days                         0
Income                              0
Count_3.6_months_late               0
Count_6.12_months_late              0
Count_more_than_12_months_late      0
application_underwriting_score      0
no_of_premiums_paid                 0
sourcing_channel                    0
residence_area_type                 0
premium                             0
dtype: int64

In [50]:
# missing data handled using XGBoost because of non normalized data and categorical features

In [51]:
# feature analysis

In [52]:
from sklearn.feature_selection import mutual_info_classif
def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_classif(X, y, discrete_features=discrete_features,n_neighbors=100)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

In [53]:
from sklearn.utils import resample
df_train_majority = df_train_completed[df_train_completed['renewal'] == 1]
df_train_minority = df_train_completed[df_train_completed['renewal'] == 0]
df_train_minority_oversampled = resample(df_train_minority, replace= True, n_samples= len(df_train_majority), random_state= 2024)
df_train_oversampled = pd.concat([df_train_majority, df_train_minority_oversampled])

In [54]:
from sklearn.preprocessing import LabelEncoder
x = df_train_oversampled.drop('renewal', axis = 1)
y = df_train_oversampled['renewal']
le = LabelEncoder()
le.fit(x['sourcing_channel'])
x['sourcing_channel_encoded'] = le.transform(x['sourcing_channel'])
x['sourcing_channel_encoded'] = x['sourcing_channel_encoded'].astype('category')
le.fit(x['residence_area_type'])
x['residence_area_type_encoded'] = le.transform(x['residence_area_type'])
x['residence_area_type_encoded'] = x['residence_area_type_encoded'].astype('category')
x = x.drop(['residence_area_type','sourcing_channel'], axis = 1)
discrete_features = [False,False,False,True,True,True,False,True,False,True,True]
make_mi_scores (x,y,discrete_features)

perc_premium_paid_by_cash_credit    0.129146
Count_6.12_months_late              0.075681
Count_3.6_months_late               0.075253
Count_more_than_12_months_late      0.054040
age_in_days                         0.050896
Income                              0.048455
application_underwriting_score      0.028017
no_of_premiums_paid                 0.016661
premium                             0.006064
sourcing_channel_encoded            0.004075
residence_area_type_encoded         0.000061
Name: MI Scores, dtype: float64

In [55]:
# the scores of mutual_info have risen after oversampling
# we will leave it here for now. next step might be PCA but we need to do a proper oversampling on em.
# we continue with xgboost for prediction.

In [142]:
df_train_le = df_train_completed.copy()
df_test_le = df_test_completed.copy()
le = LabelEncoder()

le.fit(df_train_le['sourcing_channel'])

df_train_le['sourcing_channel_encoded'] = le.transform(df_train_le['sourcing_channel'])
df_train_le['sourcing_channel_encoded'] = df_train_le['sourcing_channel_encoded'].astype('category')

df_test_le['sourcing_channel_encoded'] = le.transform(df_test_le['sourcing_channel'])
df_test_le['sourcing_channel_encoded'] = df_test_le['sourcing_channel_encoded'].astype('category')

le.fit(df_train_le['residence_area_type'])
df_train_le['residence_area_type_encoded'] = le.transform(df_train_le['residence_area_type'])
df_train_le['residence_area_type_encoded'] = df_train_le['residence_area_type_encoded'].astype('category')

df_test_le['residence_area_type_encoded'] = le.transform(df_test_le['residence_area_type'])
df_test_le['residence_area_type_encoded'] = df_test_le['residence_area_type_encoded'].astype('category')

df_train_le = df_train_le.drop(['residence_area_type','sourcing_channel'], axis = 1)
df_test_le = df_test_le.drop(['residence_area_type','sourcing_channel'], axis = 1)

In [63]:
from sklearn.model_selection import cross_val_predict, cross_val_score, StratifiedKFold, GridSearchCV
import xgboost as xgb
from sklearn.metrics import f1_score, make_scorer
my_scorer = make_scorer(f1_score, greater_is_better=True,  pos_label=0)

x = df_train_le.drop('renewal', axis = 1)
y = df_train_le['renewal']

model = xgb.XGBClassifier()
skf = StratifiedKFold(n_splits=5)
par = {
       'booster': ['gbtree'],
       'max_depth': [5,6],
       'min_child_weight': [2,3,4],
       'gamma': [7],
       'subsample': [0.8],
       'colsample_bytree': [0.9],
       'reg_alpha': [0.04, 0.05],
       'reg_lambda': [0.2, 0.3, 0.4],
       'scale_pos_weight': [0.2],
       'learning_rate': [0.01, 0.05, 0.1],
       'n_estimators': [500],
       'objective': ['binary:logistic'],
       'enable_categorical': [True],
       'random_state': [2024]
       }
GS = GridSearchCV(model, param_grid = par, cv = skf, scoring = my_scorer, n_jobs = -1)
GS.fit(x,y)
print(GS.best_score_)

0.39758677944709353


In [64]:
print(GS.best_score_)
print(GS.cv_results_['mean_test_score'])
print(GS.cv_results_['rank_test_score'])
print(GS.cv_results_['param_reg_lambda'])
print(GS.cv_results_['param_learning_rate'])
print(GS.cv_results_['param_learning_rate'])
print(GS.best_params_)

0.39758677944709353
[0.39521296 0.39600695 0.39600674 0.39503202 0.39599839 0.39517314
 0.39547903 0.39575957 0.3953323  0.39565156 0.39591409 0.39522548
 0.39563043 0.39512057 0.39553353 0.39587396 0.39547997 0.39586637
 0.39502848 0.39446905 0.3942763  0.3946686  0.39430975 0.39508451
 0.39548297 0.39487765 0.39506587 0.3954313  0.39523779 0.39502676
 0.39528184 0.39504369 0.39451328 0.39523567 0.39518466 0.39468848
 0.39524247 0.39578766 0.39563607 0.39533224 0.39507747 0.39583392
 0.39320767 0.39523259 0.39471877 0.39524818 0.39657472 0.39628066
 0.39608836 0.39568874 0.3947307  0.39558308 0.39578817 0.39532258
 0.39556641 0.3949435  0.39634941 0.39576633 0.39448738 0.39526579
 0.39595193 0.39501698 0.39330497 0.39426713 0.3956134  0.39422527
 0.39734624 0.39516216 0.39505331 0.3960176  0.39679925 0.3955207
 0.39352804 0.39482198 0.39563689 0.39352804 0.39410117 0.39690179
 0.39276334 0.39550531 0.39513754 0.39332375 0.39482251 0.39687103
 0.3941337  0.39416839 0.39337972 0.3940827

In [65]:
df_train_log = df_train_le.copy()
df_train_log['no_of_premiums_paid'] = np.log(df_train_log['no_of_premiums_paid'])
df_train_log['Income'] = np.log(df_train_log['Income'])
df_train_log['premium'] = np.log(df_train_log['premium'])

best parameters so far. no outliers were handled.
changing income into logaritmic made no difference.
{'booster': 'gbtree', 'colsample_bytree': 0.9, 'enable_categorical': True, 'gamma': 7, 'learning_rate': 0.1, 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 500, 'objective': 'binary:logistic', 'random_state': 2024, 'reg_alpha': 0.04, 'reg_lambda': 0.2, 'scale_pos_weight': 0.2, 'subsample': 0.8}

In [66]:
#trying some LOF to see the result of xgboost

In [67]:
from sklearn.neighbors import LocalOutlierFactor
lof = LocalOutlierFactor(n_neighbors=100)
y_outlier = lof.fit_predict(df_train_log)
y_outlier = pd.DataFrame(y_outlier, columns=['y_outlier'])
y_outlier

,y_outlier
0,1
1,-1
2,1
3,1
4,1
...,...
53231,1
53232,1
53233,1
53234,1


In [68]:
df_train_lof = df_train_log.copy()
df_train_lof['lof'] = y_outlier
df_train_lof

,perc_premium_paid_by_cash_credit,age_in_days,Income,Count_3.6_months_late,Count_6.12_months_late,Count_more_than_12_months_late,application_underwriting_score,no_of_premiums_paid,premium,renewal,sourcing_channel_encoded,residence_area_type_encoded,lof
0,0.429,12058,12.780042,0.0,0.0,0.0,99.02,2.564949,8.101678,1,2,1,1
1,0.917,17531,11.340237,2.0,3.0,1.0,98.69,1.945910,8.101678,0,2,0,-1
2,0.049,15341,12.431254,0.0,0.0,0.0,99.57,2.197225,9.169518,1,0,1,1
3,0.052,31400,12.199451,0.0,0.0,0.0,99.87,2.484907,9.169518,1,1,1,1
4,1.000,24829,11.681824,0.0,0.0,0.0,99.05,2.397895,8.922658,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
53231,0.994,20445,12.137954,0.0,0.0,0.0,99.67,1.609438,9.798127,1,0,1,1
53232,0.118,22275,12.181114,0.0,0.0,0.0,99.25,2.397895,9.169518,1,0,1,1
53233,0.033,18265,12.616658,0.0,0.0,0.0,99.89,1.386294,9.532424,1,0,0,1
53234,0.000,23372,12.628133,0.0,0.0,0.0,98.89,2.484907,9.169518,1,0,0,1


In [69]:
df_train_drop_outlier = df_train_lof.copy()
df_train_drop_outlier = df_train_drop_outlier[df_train_drop_outlier['lof'] == 1].drop('lof', axis = 1)
df_train_drop_outlier

,perc_premium_paid_by_cash_credit,age_in_days,Income,Count_3.6_months_late,Count_6.12_months_late,Count_more_than_12_months_late,application_underwriting_score,no_of_premiums_paid,premium,renewal,sourcing_channel_encoded,residence_area_type_encoded
0,0.429,12058,12.780042,0.0,0.0,0.0,99.02,2.564949,8.101678,1,2,1
2,0.049,15341,12.431254,0.0,0.0,0.0,99.57,2.197225,9.169518,1,0,1
3,0.052,31400,12.199451,0.0,0.0,0.0,99.87,2.484907,9.169518,1,1,1
4,1.000,24829,11.681824,0.0,0.0,0.0,99.05,2.397895,8.922658,1,1,1
6,0.621,9868,11.435180,0.0,0.0,0.0,99.58,1.386294,8.922658,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
53231,0.994,20445,12.137954,0.0,0.0,0.0,99.67,1.609438,9.798127,1,0,1
53232,0.118,22275,12.181114,0.0,0.0,0.0,99.25,2.397895,9.169518,1,0,1
53233,0.033,18265,12.616658,0.0,0.0,0.0,99.89,1.386294,9.532424,1,0,0
53234,0.000,23372,12.628133,0.0,0.0,0.0,98.89,2.484907,9.169518,1,0,0


In [74]:
x = df_train_drop_outlier.drop('renewal', axis = 1)
y = df_train_drop_outlier['renewal']

model = xgb.XGBClassifier()
skf = StratifiedKFold(n_splits=5)
par = {
       'booster': ['gbtree'],
       'max_depth': [5,6],
       'min_child_weight': [2,3],
       'gamma': [0.1],
       'subsample': [0.8],
       'colsample_bytree': [0.9],
       'reg_alpha': [0.05],
       'reg_lambda': [0.3, 0.4, 0.5],
       'scale_pos_weight': [0.2],
       'learning_rate': [0.1],
       'n_estimators': [100],
       'objective': ['binary:logistic'],
       'enable_categorical': [True],
       'random_state': [2024]
       }
GS = GridSearchCV(model, param_grid = par, cv = skf, scoring = my_scorer, n_jobs = -1)
GS.fit(x,y)
print(GS.best_score_)
print(GS.best_params_)

0.31134959328078293
{'booster': 'gbtree', 'colsample_bytree': 0.9, 'enable_categorical': True, 'gamma': 0.1, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 100, 'objective': 'binary:logistic', 'random_state': 2024, 'reg_alpha': 0.05, 'reg_lambda': 0.5, 'scale_pos_weight': 0.2, 'subsample': 0.8}


In [432]:
#local outlier factor did not do any good

In [ ]:
# -------------------------

In [76]:
# trying gblinear boosting method instead of gbtree.
# it seems it does not support categorical features. I need to do dummy encoding for it.
df_train_dummy = df_train_completed.copy()
df_train_dummy = df_train_dummy.join(pd.get_dummies(df_train_dummy['sourcing_channel'])).drop(['sourcing_channel','E'],axis =1)
df_train_dummy = df_train_dummy.join(pd.get_dummies(df_train_dummy['residence_area_type'])).drop(['residence_area_type','Rural'],axis =1)
df_train_dummy

,perc_premium_paid_by_cash_credit,age_in_days,Income,Count_3.6_months_late,Count_6.12_months_late,Count_more_than_12_months_late,application_underwriting_score,no_of_premiums_paid,premium,renewal,A,B,C,D,Urban
0,0.429,12058,355060,0.0,0.0,0.0,99.02,13,3300,1,0,0,1,0,1
1,0.917,17531,84140,2.0,3.0,1.0,98.69,7,3300,0,0,0,1,0,0
2,0.049,15341,250510,0.0,0.0,0.0,99.57,9,9600,1,1,0,0,0,1
3,0.052,31400,198680,0.0,0.0,0.0,99.87,12,9600,1,0,1,0,0,1
4,1.000,24829,118400,0.0,0.0,0.0,99.05,11,7500,1,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53231,0.994,20445,186830,0.0,0.0,0.0,99.67,5,18000,1,1,0,0,0,1
53232,0.118,22275,195070,0.0,0.0,0.0,99.25,11,9600,1,1,0,0,0,1
53233,0.033,18265,301540,0.0,0.0,0.0,99.89,4,13800,1,1,0,0,0,0
53234,0.000,23372,305020,0.0,0.0,0.0,98.89,12,9600,1,1,0,0,0,0


In [78]:
x = df_train_dummy.drop('renewal', axis = 1)
y = df_train_dummy['renewal']

model = xgb.XGBClassifier()
skf = StratifiedKFold(n_splits=5)
par = {
       'booster': ['gblinear'],
       'reg_alpha': [0, 0.01, 0.1],
       'reg_lambda': [0, 0.1, 1],
       'scale_pos_weight': [0.1, 0.15, 0.2],
       'learning_rate': [0.01, 0.03, 0.05, 0.7],
       'n_estimators': [100],
       'objective': ['binary:logistic'],
       'enable_categorical': [True],
       'random_state': [2024]
       }
GS = GridSearchCV(model, param_grid = par, cv = skf, scoring = my_scorer, n_jobs = -1)
GS.fit(x,y)
print(GS.best_score_)
print(GS.best_params_)

0.38304265619012934
{'booster': 'gblinear', 'enable_categorical': True, 'learning_rate': 0.03, 'n_estimators': 100, 'objective': 'binary:logistic', 'random_state': 2024, 'reg_alpha': 0, 'reg_lambda': 0, 'scale_pos_weight': 0.15}


In [79]:
#training main model based on first gridsearch

{'booster': 'gbtree', 'colsample_bytree': 0.9, 'enable_categorical': True, 'gamma': 7, 'learning_rate': 0.1, 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 500, 'objective': 'binary:logistic', 'random_state': 2024, 'reg_alpha': 0.04, 'reg_lambda': 0.2, 'scale_pos_weight': 0.2, 'subsample': 0.8}

In [137]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
x = df_train_le.drop('renewal', axis = 1)
y = df_train_le['renewal']
model = xgb.XGBClassifier(booster = 'gbtree', colsample_bytree = 0.9, enable_categorical = True, gamma = 7,
                          learning_rate = 0.1, max_depth = 6, min_child_weight = 3, n_estimators = 500, 
                          reg_alpha = 0.04, reg_lambda = 0.2, scale_pos_weight = 0.2, subsample = 0.8, random_state = 2024)
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 2024)
my_scorer = make_scorer(f1_score, greater_is_better=True,  pos_label=0)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=2024)
scores = cross_val_score(model, x_train, y_train, scoring = my_scorer, cv = skf, n_jobs=-1)

In [138]:
scores

array([0.40630182, 0.38796516, 0.39967768, 0.39059968, 0.37928287])

In [139]:
from sklearn.metrics import f1_score, make_scorer
model.fit(x_train, y_train)
yhat = model.predict(x_val)
f1_score(y_val, yhat, pos_label = 0)

0.3815028901734104

In [145]:
model.fit(x, y)
Y_final_prediction = pd.DataFrame(model.predict(df_test_le))

In [147]:
Y_final_prediction.value_counts()

1    24303
0     2314
dtype: int64

In [149]:
Y_final_prediction.to_csv('prediction2.csv',index=False,header=None)

معین نجفی زاده	۰۶ دی ۱۴۰۳ ساعت ۱۹:۰۱	۰.۴۱۲۱